# Tri-Lobed Ring Drone Family — Full Colab Prototype Notebook

This notebook builds a **full prototype family** for a **tri-lobed ring drone with perimeter micro-nozzle belts**.

It is designed to run **entirely in Google Colab** and includes:

- parametric **tri-lobed geometry**
- **multiple size classes** and **variation families**
- **pure fluidic**, **compressor-fed**, and **hybrid central-lift** variants
- nozzle-belt layout synthesis
- coarse **thrust / power / authority / endurance** estimators
- **design sweeps**
- **ranking / trade studies**
- geometry visualization
- result export to **CSV / JSON / PNG / ZIP**

## Design intent

This notebook is not pretending to be CFD. It is an **engineering exploration scaffold** that lets you:

1. generate a large family of candidate vehicles,
2. compare them under a consistent scoring model,
3. inspect geometry and nozzle placement,
4. export bundles for further iteration.

The **most realistic near-term path** in this notebook is the **hybrid architecture**:
- central lift handles most weight support,
- ring micro-nozzles handle control augmentation, yaw, gust rejection, lateral trim, and transients.

Run the notebook top-to-bottom, then tune the configuration section and rerun.

In [ ]:
# ============================================================
# 0. Environment + deterministic setup
# ============================================================
import os
import io
import json
import math
import time
import zipfile
import pathlib
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)

OUT_DIR = pathlib.Path("tri_lobed_ring_drone_family_out")
OUT_DIR.mkdir(exist_ok=True, parents=True)

print("OUT_DIR:", OUT_DIR.resolve())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

In [ ]:
# ============================================================
# 1. Global configuration
# ============================================================
# Units:
# - length: meters
# - mass: kilograms
# - force: Newtons
# - power: Watts
# - pressure: Pascals
# - angles: radians unless stated otherwise

RHO_AIR = 1.225
G = 9.80665
ETA_COMPRESSOR_DEFAULT = 0.46
ETA_DUCTED_FAN_DEFAULT = 0.68
BATTERY_WH_PER_KG = 220.0  # coarse system-level battery specific energy
C_DISCHARGE_USABLE = 0.82  # only usable fraction under load / reserve margin

# High-level design assumptions:
# 1) Fluidic micro-nozzles are less hover-efficient than rotor/ducted-fan lift.
# 2) Hybrid architectures are favored in score because they are more likely to be practical.
# 3) Pressure-fed systems face manifold and valve penalties as nozzle count increases.

SWEEP_CFG = {
    "diameters_m": [0.18, 0.32, 0.62, 1.40],
    "lobe_depth_ratio": [0.14, 0.18, 0.22],
    "section_thickness_ratio": [0.10, 0.12, 0.14],
    "section_height_ratio": [0.08, 0.10, 0.12],
    "nozzle_types": ["round", "slot", "mixed"],
    "architectures": ["cold_gas", "compressor_fluidic", "hybrid_central_lift"],
    "sector_counts": [12, 24, 36, 60],
    "lift_split_central": [0.00, 0.70, 0.80, 0.85, 0.90],  # fraction of lift from central system
    "round_nozzle_diams_mm": [0.4, 0.6, 0.8, 1.2],
    "slot_lengths_mm": [3.0, 5.0, 8.0],
    "slot_widths_mm": [0.3, 0.5, 0.8],
    "plenum_pressure_kpa_gauge": [8.0, 15.0, 25.0, 40.0],  # gauge pressure
}

print(json.dumps(SWEEP_CFG, indent=2))

In [ ]:
# ============================================================
# 2. Data models
# ============================================================
@dataclass
class PrototypeSpec:
    name: str
    size_class: str
    architecture: str  # cold_gas, compressor_fluidic, hybrid_central_lift
    diameter_m: float
    lobe_depth_ratio: float
    section_thickness_ratio: float
    section_height_ratio: float
    nozzle_type: str  # round, slot, mixed
    sector_count: int
    lower_belt_fraction: float
    upper_belt_fraction: float
    tangential_belt_fraction: float
    round_nozzle_diam_mm: float
    slot_length_mm: float
    slot_width_mm: float
    plenum_pressure_kpa_gauge: float
    lift_split_central: float
    battery_mass_kg: float
    avionics_mass_kg: float
    structure_mass_factor: float = 1.0
    payload_mass_kg: float = 0.0

@dataclass
class GeometryPack:
    outline_xy: np.ndarray
    inner_xy: np.ndarray
    nozzle_positions_xy: np.ndarray
    nozzle_dirs_xy: np.ndarray
    nozzle_labels: List[str]
    metrics: Dict[str, float]

@dataclass
class PerformanceResult:
    thrust_fluidic_N: float
    thrust_central_N: float
    thrust_total_N: float
    hover_margin: float
    gross_mass_kg: float
    usable_energy_Wh: float
    hover_power_W: float
    endurance_min: float
    control_authority_score: float
    yaw_authority_score: float
    fault_tolerance_score: float
    manufacturability_score: float
    practical_score: float
    total_score: float
    notes: Dict[str, float] = field(default_factory=dict)

In [ ]:
# ============================================================
# 3. Geometry engine
# ============================================================
def tri_lobed_radius(theta: np.ndarray, base_r: float, lobe_amp: float) -> np.ndarray:
    # Tri-lobed polar radius model; smooth and easy to parameterize.
    return base_r * (1.0 + lobe_amp * np.cos(3.0 * theta))

def smooth_xy_from_polar(theta: np.ndarray, r: np.ndarray) -> np.ndarray:
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return np.stack([x, y], axis=1)

def polygon_length(xy: np.ndarray) -> float:
    d = np.diff(np.vstack([xy, xy[:1]]), axis=0)
    return float(np.sum(np.linalg.norm(d, axis=1)))

def resample_polyline_closed(xy: np.ndarray, n: int) -> np.ndarray:
    pts = np.vstack([xy, xy[:1]])
    seg = np.linalg.norm(np.diff(pts, axis=0), axis=1)
    s = np.concatenate([[0.0], np.cumsum(seg)])
    total = s[-1]
    targets = np.linspace(0.0, total, n + 1)[:-1]
    out = []
    j = 0
    for t in targets:
        while j < len(seg) - 1 and s[j + 1] < t:
            j += 1
        denom = max(1e-12, s[j+1] - s[j])
        a = (t - s[j]) / denom
        p = pts[j] * (1 - a) + pts[j + 1] * a
        out.append(p)
    return np.asarray(out)

def normals_for_closed_polyline(xy: np.ndarray) -> np.ndarray:
    pts_prev = np.roll(xy, 1, axis=0)
    pts_next = np.roll(xy, -1, axis=0)
    tang = pts_next - pts_prev
    tang = tang / np.clip(np.linalg.norm(tang, axis=1, keepdims=True), 1e-12, None)
    normals = np.stack([-tang[:,1], tang[:,0]], axis=1)
    return normals

def generate_tri_lobed_geometry(
    diameter_m: float,
    lobe_depth_ratio: float,
    section_thickness_ratio: float,
    section_height_ratio: float,
    sector_count: int,
    lower_belt_fraction: float,
    upper_belt_fraction: float,
    tangential_belt_fraction: float,
) -> GeometryPack:
    ro = diameter_m / 2.0
    theta = np.linspace(0.0, 2.0 * np.pi, 1200, endpoint=False)

    # Convert the user's intuitive ratios into a smooth tri-lobed ring.
    base_r = ro * (1.0 - 0.55 * lobe_depth_ratio)
    lobe_amp = min(0.35, max(0.05, lobe_depth_ratio))
    outer_r = tri_lobed_radius(theta, base_r, lobe_amp)

    # Inner shell: simple radial offset; enough for conceptual geometry.
    ring_t = diameter_m * section_thickness_ratio
    inner_r = np.maximum(outer_r - ring_t, outer_r * 0.35)

    outline_xy = smooth_xy_from_polar(theta, outer_r)
    inner_xy = smooth_xy_from_polar(theta, inner_r)

    # Sector center placement
    sector_pts = resample_polyline_closed(outline_xy, sector_count)
    sector_normals = normals_for_closed_polyline(sector_pts)
    tangents = np.stack([sector_normals[:,1], -sector_normals[:,0]], axis=1)

    belts = []
    dirs = []
    labels = []

    counts = {
        "lower": max(1, int(round(sector_count * lower_belt_fraction))),
        "upper": max(1, int(round(sector_count * upper_belt_fraction))),
        "tangential": max(1, int(round(sector_count * tangential_belt_fraction))),
    }

    # Use every sector for all belts conceptually; the fractions scale force weighting later.
    for i, p in enumerate(sector_pts):
        n = sector_normals[i]
        t = tangents[i]
        belts.append(p - 0.008 * n)   # lower
        dirs.append(-n)
        labels.append("lower")

        belts.append(p + 0.008 * n)   # upper
        dirs.append(+n)
        labels.append("upper")

        belts.append(p)               # tangential
        dirs.append(t)
        labels.append("tangential")

    nozzle_positions_xy = np.asarray(belts)
    nozzle_dirs_xy = np.asarray(dirs)

    perimeter_m = polygon_length(outline_xy)
    metrics = {
        "outer_perimeter_m": perimeter_m,
        "mean_outer_radius_m": float(np.mean(np.linalg.norm(outline_xy, axis=1))),
        "section_height_m": diameter_m * section_height_ratio,
        "ring_thickness_m": ring_t,
        "approx_planform_area_m2": float(np.pi * (ro ** 2) * 0.72),
        "aperture_area_m2": float(max(0.0, np.pi * (np.mean(inner_r) ** 2) * 0.85)),
    }

    return GeometryPack(
        outline_xy=outline_xy,
        inner_xy=inner_xy,
        nozzle_positions_xy=nozzle_positions_xy,
        nozzle_dirs_xy=nozzle_dirs_xy,
        nozzle_labels=labels,
        metrics=metrics,
    )

In [ ]:
# ============================================================
# 4. Mass models
# ============================================================
def size_class_from_diameter(d: float) -> str:
    if d <= 0.22:
        return "P0"
    if d <= 0.40:
        return "P1"
    if d <= 0.90:
        return "P2"
    return "P3"

def estimate_structure_mass_kg(spec: PrototypeSpec, geom: GeometryPack) -> float:
    # Coarse scaling: shell mass roughly follows perimeter * section area * material factor.
    perimeter = geom.metrics["outer_perimeter_m"]
    section_h = geom.metrics["section_height_m"]
    ring_t = geom.metrics["ring_thickness_m"]

    raw = perimeter * section_h * ring_t * 85.0  # tuned conceptual density factor
    architecture_factor = {
        "cold_gas": 0.85,
        "compressor_fluidic": 1.0,
        "hybrid_central_lift": 1.22,
    }[spec.architecture]
    return raw * architecture_factor * spec.structure_mass_factor

def estimate_propulsion_aux_mass_kg(spec: PrototypeSpec, geom: GeometryPack) -> float:
    d = spec.diameter_m
    if spec.architecture == "cold_gas":
        return 0.12 + 0.35 * d
    if spec.architecture == "compressor_fluidic":
        return 0.22 + 0.95 * d
    if spec.architecture == "hybrid_central_lift":
        return 0.40 + 1.65 * d
    return 0.0

def estimate_gross_mass_kg(spec: PrototypeSpec, geom: GeometryPack) -> float:
    structure = estimate_structure_mass_kg(spec, geom)
    aux = estimate_propulsion_aux_mass_kg(spec, geom)
    gross = structure + aux + spec.battery_mass_kg + spec.avionics_mass_kg + spec.payload_mass_kg
    return gross

In [ ]:
# ============================================================
# 5. Fluidic + hybrid performance estimators
# ============================================================
def nozzle_area_m2(spec: PrototypeSpec) -> float:
    if spec.nozzle_type == "round":
        d = spec.round_nozzle_diam_mm * 1e-3
        return math.pi * (d * 0.5) ** 2
    if spec.nozzle_type == "slot":
        return (spec.slot_length_mm * 1e-3) * (spec.slot_width_mm * 1e-3)
    # mixed
    d = spec.round_nozzle_diam_mm * 1e-3
    round_a = math.pi * (d * 0.5) ** 2
    slot_a = (spec.slot_length_mm * 1e-3) * (spec.slot_width_mm * 1e-3)
    return 0.5 * (round_a + slot_a)

def nozzle_count_total(spec: PrototypeSpec) -> int:
    # 3 belt placements per sector in this notebook
    return int(spec.sector_count * 3)

def valve_penalty(spec: PrototypeSpec) -> float:
    n = nozzle_count_total(spec)
    # More actuators = better spatial authority, but more loss / complexity.
    return 1.0 / (1.0 + 0.0025 * max(0, n - 36))

def manifold_penalty(spec: PrototypeSpec, geom: GeometryPack) -> float:
    perimeter = geom.metrics["outer_perimeter_m"]
    pressure = spec.plenum_pressure_kpa_gauge
    # Pressure distribution gets harder at large perimeter / high count.
    loss = 1.0 / (1.0 + 0.08 * perimeter + 0.01 * spec.sector_count + 0.002 * pressure)
    return float(loss)

def nozzle_type_efficiency_multiplier(nozzle_type: str) -> float:
    return {
        "round": 0.97,
        "slot": 0.91,
        "mixed": 1.00,
    }[nozzle_type]

def fluidic_thrust_N(spec: PrototypeSpec, geom: GeometryPack) -> float:
    # Very coarse jet thrust estimate using pressure * area with multiple penalties.
    A = nozzle_area_m2(spec)
    Nn = nozzle_count_total(spec)
    dP = spec.plenum_pressure_kpa_gauge * 1000.0
    base = dP * A * Nn

    belt_weight = (
        1.00 * spec.lower_belt_fraction +
        0.72 * spec.upper_belt_fraction +
        0.78 * spec.tangential_belt_fraction
    )
    base *= belt_weight

    # Additional scaling to keep conceptual outputs in plausible ranges.
    base *= 2.1

    penalties = (
        valve_penalty(spec) *
        manifold_penalty(spec, geom) *
        nozzle_type_efficiency_multiplier(spec.nozzle_type)
    )

    arch_mult = {
        "cold_gas": 1.00,
        "compressor_fluidic": 0.92,
        "hybrid_central_lift": 0.82,  # ring nozzles often sized more for control than main lift
    }[spec.architecture]

    return float(base * penalties * arch_mult)

def central_lift_thrust_N(spec: PrototypeSpec, geom: GeometryPack, gross_mass_kg: float) -> float:
    if spec.architecture != "hybrid_central_lift":
        return 0.0
    # In hybrid mode we intentionally size central system to carry requested lift split plus margin.
    return gross_mass_kg * G * spec.lift_split_central * 1.10

def fluidic_power_W(spec: PrototypeSpec, geom: GeometryPack, thrust_fluidic_N: float) -> float:
    dP = spec.plenum_pressure_kpa_gauge * 1000.0
    A_total = nozzle_area_m2(spec) * nozzle_count_total(spec)
    Cd = 0.84
    v_exit = math.sqrt(max(1e-9, 2.0 * dP / RHO_AIR))
    q = Cd * A_total * v_exit  # m^3/s
    p_shaft = (dP * q) / max(1e-6, ETA_COMPRESSOR_DEFAULT)

    # Architecture scaling
    if spec.architecture == "cold_gas":
        p_shaft *= 0.15  # no compressor onboard in static model; represent low electrical demand
    elif spec.architecture == "hybrid_central_lift":
        p_shaft *= 0.72

    # Add valve and control electronics
    p_shaft += 1.2 * spec.sector_count
    return float(p_shaft)

def central_lift_power_W(spec: PrototypeSpec, geom: GeometryPack, thrust_central_N: float) -> float:
    if thrust_central_N <= 0.0:
        return 0.0
    disk_area = max(1e-4, geom.metrics["aperture_area_m2"] * 0.78)
    induced = (thrust_central_N ** 1.5) / math.sqrt(2.0 * RHO_AIR * disk_area)
    return float(induced / max(1e-6, ETA_DUCTED_FAN_DEFAULT) * 1.18)

def control_authority_score(spec: PrototypeSpec, geom: GeometryPack, thrust_fluidic_N: float, gross_mass_kg: float) -> float:
    leverage = geom.metrics["mean_outer_radius_m"]
    ratio = thrust_fluidic_N * leverage / max(1e-6, gross_mass_kg * G * geom.metrics["mean_outer_radius_m"])
    sector_bonus = 1.0 - math.exp(-spec.sector_count / 18.0)
    tangential_bonus = 0.55 + 0.9 * spec.tangential_belt_fraction
    return float(100.0 * min(1.25, ratio * 1.8) * sector_bonus * tangential_bonus)

def yaw_authority_score(spec: PrototypeSpec, geom: GeometryPack, thrust_fluidic_N: float) -> float:
    leverage = geom.metrics["mean_outer_radius_m"]
    yaw = thrust_fluidic_N * leverage * (0.4 + spec.tangential_belt_fraction)
    scale = 220.0 * (0.4 + geom.metrics["mean_outer_radius_m"])
    return float(100.0 * (1.0 - math.exp(-yaw / max(1e-6, scale))))

def fault_tolerance_score(spec: PrototypeSpec) -> float:
    n = nozzle_count_total(spec)
    sectors = spec.sector_count
    arch_bonus = {"cold_gas": 0.72, "compressor_fluidic": 0.82, "hybrid_central_lift": 1.0}[spec.architecture]
    return float(min(100.0, arch_bonus * (40.0 + 10.0 * math.log1p(n) + 0.5 * sectors)))

def manufacturability_score(spec: PrototypeSpec) -> float:
    complexity = (
        0.6 * nozzle_count_total(spec) +
        0.9 * spec.sector_count +
        (18.0 if spec.nozzle_type == "mixed" else 10.0 if spec.nozzle_type == "slot" else 6.0)
    )
    arch_penalty = {"cold_gas": 8.0, "compressor_fluidic": 16.0, "hybrid_central_lift": 28.0}[spec.architecture]
    score = 100.0 - 0.35 * complexity - arch_penalty
    return float(max(5.0, score))

def practical_score(spec: PrototypeSpec, hover_margin: float, endurance_min: float) -> float:
    margin_score = 100.0 * np.clip((hover_margin - 0.85) / 0.45, 0.0, 1.0)
    endurance_score = 100.0 * np.clip(endurance_min / 24.0, 0.0, 1.0)
    arch_bias = {"cold_gas": 0.35, "compressor_fluidic": 0.65, "hybrid_central_lift": 1.00}[spec.architecture]
    return float((0.58 * margin_score + 0.42 * endurance_score) * arch_bias)

def evaluate_prototype(spec: PrototypeSpec) -> Tuple[GeometryPack, PerformanceResult]:
    geom = generate_tri_lobed_geometry(
        diameter_m=spec.diameter_m,
        lobe_depth_ratio=spec.lobe_depth_ratio,
        section_thickness_ratio=spec.section_thickness_ratio,
        section_height_ratio=spec.section_height_ratio,
        sector_count=spec.sector_count,
        lower_belt_fraction=spec.lower_belt_fraction,
        upper_belt_fraction=spec.upper_belt_fraction,
        tangential_belt_fraction=spec.tangential_belt_fraction,
    )

    gross_mass = estimate_gross_mass_kg(spec, geom)
    fluidic = fluidic_thrust_N(spec, geom)
    central = central_lift_thrust_N(spec, geom, gross_mass)
    total = fluidic + central

    hover_required = gross_mass * G
    hover_margin = total / max(1e-6, hover_required)

    usable_energy_Wh = spec.battery_mass_kg * BATTERY_WH_PER_KG * C_DISCHARGE_USABLE
    p_fluidic = fluidic_power_W(spec, geom, fluidic)
    p_central = central_lift_power_W(spec, geom, central)
    hover_power = p_fluidic + p_central + 12.0  # avionics reserve
    endurance_min = 60.0 * usable_energy_Wh / max(1e-6, hover_power)

    ctrl = control_authority_score(spec, geom, fluidic, gross_mass)
    yaw = yaw_authority_score(spec, geom, fluidic)
    fault = fault_tolerance_score(spec)
    manu = manufacturability_score(spec)
    prac = practical_score(spec, hover_margin, endurance_min)

    # Weighted score intentionally favors practical hybrid designs but still rewards control novelty.
    total_score = (
        0.24 * ctrl +
        0.12 * yaw +
        0.16 * fault +
        0.10 * manu +
        0.38 * prac
    )

    notes = {
        "fluidic_power_W": p_fluidic,
        "central_power_W": p_central,
        "outer_perimeter_m": geom.metrics["outer_perimeter_m"],
        "aperture_area_m2": geom.metrics["aperture_area_m2"],
        "nozzle_area_m2": nozzle_area_m2(spec),
        "nozzle_count_total": nozzle_count_total(spec),
    }

    perf = PerformanceResult(
        thrust_fluidic_N=fluidic,
        thrust_central_N=central,
        thrust_total_N=total,
        hover_margin=hover_margin,
        gross_mass_kg=gross_mass,
        usable_energy_Wh=usable_energy_Wh,
        hover_power_W=hover_power,
        endurance_min=endurance_min,
        control_authority_score=ctrl,
        yaw_authority_score=yaw,
        fault_tolerance_score=fault,
        manufacturability_score=manu,
        practical_score=prac,
        total_score=total_score,
        notes=notes,
    )
    return geom, perf

In [ ]:
# ============================================================
# 6. Named prototype family presets
# ============================================================
def preset_family() -> List[PrototypeSpec]:
    presets = []

    # P0
    presets.append(PrototypeSpec(
        name="P0-A-36R",
        size_class="P0",
        architecture="cold_gas",
        diameter_m=0.18,
        lobe_depth_ratio=0.18,
        section_thickness_ratio=0.12,
        section_height_ratio=0.10,
        nozzle_type="round",
        sector_count=12,
        lower_belt_fraction=0.34,
        upper_belt_fraction=0.33,
        tangential_belt_fraction=0.33,
        round_nozzle_diam_mm=0.6,
        slot_length_mm=5.0,
        slot_width_mm=0.5,
        plenum_pressure_kpa_gauge=25.0,
        lift_split_central=0.0,
        battery_mass_kg=0.05,
        avionics_mass_kg=0.04,
    ))
    presets.append(PrototypeSpec(
        name="P0-C-36M",
        size_class="P0",
        architecture="compressor_fluidic",
        diameter_m=0.18,
        lobe_depth_ratio=0.22,
        section_thickness_ratio=0.12,
        section_height_ratio=0.10,
        nozzle_type="mixed",
        sector_count=12,
        lower_belt_fraction=0.40,
        upper_belt_fraction=0.25,
        tangential_belt_fraction=0.35,
        round_nozzle_diam_mm=0.8,
        slot_length_mm=5.0,
        slot_width_mm=0.5,
        plenum_pressure_kpa_gauge=15.0,
        lift_split_central=0.0,
        battery_mass_kg=0.08,
        avionics_mass_kg=0.05,
    ))

    # P1
    presets.append(PrototypeSpec(
        name="P1-A-72F",
        size_class="P1",
        architecture="compressor_fluidic",
        diameter_m=0.32,
        lobe_depth_ratio=0.18,
        section_thickness_ratio=0.12,
        section_height_ratio=0.10,
        nozzle_type="round",
        sector_count=24,
        lower_belt_fraction=0.42,
        upper_belt_fraction=0.22,
        tangential_belt_fraction=0.36,
        round_nozzle_diam_mm=0.8,
        slot_length_mm=5.0,
        slot_width_mm=0.5,
        plenum_pressure_kpa_gauge=25.0,
        lift_split_central=0.0,
        battery_mass_kg=0.22,
        avionics_mass_kg=0.08,
    ))
    presets.append(PrototypeSpec(
        name="P1-C-48H",
        size_class="P1",
        architecture="hybrid_central_lift",
        diameter_m=0.32,
        lobe_depth_ratio=0.18,
        section_thickness_ratio=0.12,
        section_height_ratio=0.10,
        nozzle_type="mixed",
        sector_count=16,
        lower_belt_fraction=0.32,
        upper_belt_fraction=0.20,
        tangential_belt_fraction=0.48,
        round_nozzle_diam_mm=0.6,
        slot_length_mm=3.0,
        slot_width_mm=0.3,
        plenum_pressure_kpa_gauge=15.0,
        lift_split_central=0.85,
        battery_mass_kg=0.28,
        avionics_mass_kg=0.10,
    ))

    # P2
    presets.append(PrototypeSpec(
        name="P2-C1-96",
        size_class="P2",
        architecture="hybrid_central_lift",
        diameter_m=0.62,
        lobe_depth_ratio=0.18,
        section_thickness_ratio=0.12,
        section_height_ratio=0.10,
        nozzle_type="round",
        sector_count=32,
        lower_belt_fraction=0.24,
        upper_belt_fraction=0.18,
        tangential_belt_fraction=0.58,
        round_nozzle_diam_mm=0.8,
        slot_length_mm=5.0,
        slot_width_mm=0.5,
        plenum_pressure_kpa_gauge=15.0,
        lift_split_central=0.90,
        battery_mass_kg=1.25,
        avionics_mass_kg=0.22,
        payload_mass_kg=0.35,
    ))
    presets.append(PrototypeSpec(
        name="P2-C2-144",
        size_class="P2",
        architecture="hybrid_central_lift",
        diameter_m=0.62,
        lobe_depth_ratio=0.22,
        section_thickness_ratio=0.14,
        section_height_ratio=0.12,
        nozzle_type="mixed",
        sector_count=48,
        lower_belt_fraction=0.40,
        upper_belt_fraction=0.18,
        tangential_belt_fraction=0.42,
        round_nozzle_diam_mm=0.8,
        slot_length_mm=5.0,
        slot_width_mm=0.5,
        plenum_pressure_kpa_gauge=25.0,
        lift_split_central=0.80,
        battery_mass_kg=1.55,
        avionics_mass_kg=0.25,
        payload_mass_kg=0.65,
    ))
    presets.append(PrototypeSpec(
        name="P2-C3-SAFE",
        size_class="P2",
        architecture="hybrid_central_lift",
        diameter_m=0.62,
        lobe_depth_ratio=0.18,
        section_thickness_ratio=0.14,
        section_height_ratio=0.12,
        nozzle_type="slot",
        sector_count=36,
        lower_belt_fraction=0.36,
        upper_belt_fraction=0.22,
        tangential_belt_fraction=0.42,
        round_nozzle_diam_mm=0.8,
        slot_length_mm=8.0,
        slot_width_mm=0.5,
        plenum_pressure_kpa_gauge=15.0,
        lift_split_central=0.85,
        battery_mass_kg=1.45,
        avionics_mass_kg=0.24,
        payload_mass_kg=0.45,
    ))

    # P3
    presets.append(PrototypeSpec(
        name="P3-CX-240",
        size_class="P3",
        architecture="hybrid_central_lift",
        diameter_m=1.40,
        lobe_depth_ratio=0.22,
        section_thickness_ratio=0.14,
        section_height_ratio=0.12,
        nozzle_type="mixed",
        sector_count=80,
        lower_belt_fraction=0.38,
        upper_belt_fraction=0.18,
        tangential_belt_fraction=0.44,
        round_nozzle_diam_mm=1.2,
        slot_length_mm=8.0,
        slot_width_mm=0.8,
        plenum_pressure_kpa_gauge=25.0,
        lift_split_central=0.85,
        battery_mass_kg=6.5,
        avionics_mass_kg=0.65,
        payload_mass_kg=2.5,
    ))
    return presets

family = preset_family()
print("Preset count:", len(family))
for s in family:
    print("-", s.name)

In [ ]:
# ============================================================
# 7. Evaluate named family
# ============================================================
rows = []
geoms_by_name = {}

for spec in family:
    geom, perf = evaluate_prototype(spec)
    geoms_by_name[spec.name] = geom
    row = {**asdict(spec), **asdict(perf)}
    # flatten notes
    for k, v in perf.notes.items():
        row[k] = v
    rows.append(row)

family_df = pd.DataFrame(rows).sort_values(["size_class", "total_score"], ascending=[True, False]).reset_index(drop=True)
family_df

In [ ]:
# Save preset family results
family_csv = OUT_DIR / "preset_family_results.csv"
family_json = OUT_DIR / "preset_family_results.json"

family_df.to_csv(family_csv, index=False)
family_df.to_json(family_json, orient="records", indent=2)

print("Saved:", family_csv)
print("Saved:", family_json)
print(family_df[[
    "name", "size_class", "architecture", "gross_mass_kg", "thrust_total_N",
    "hover_margin", "endurance_min", "control_authority_score", "total_score"
]].round(3))

In [ ]:
# ============================================================
# 8. Plot geometry helper
# ============================================================
def plot_geometry(spec: PrototypeSpec, geom: GeometryPack, ax=None, title=None, savepath=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 7))
    else:
        fig = ax.figure

    ax.plot(geom.outline_xy[:,0], geom.outline_xy[:,1], lw=2, label="outer")
    ax.plot(geom.inner_xy[:,0], geom.inner_xy[:,1], lw=1.5, label="inner")

    label_colors = {
        "lower": None,
        "upper": None,
        "tangential": None,
    }

    for label in ["lower", "upper", "tangential"]:
        idx = [i for i, lab in enumerate(geom.nozzle_labels) if lab == label]
        pts = geom.nozzle_positions_xy[idx]
        vec = geom.nozzle_dirs_xy[idx]
        ax.scatter(pts[:,0], pts[:,1], s=12, label=label)
        skip = max(1, len(idx) // 16)
        for p, v in zip(pts[::skip], vec[::skip]):
            ax.arrow(p[0], p[1], 0.02 * v[0], 0.02 * v[1], head_width=0.008, length_includes_head=True)

    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.legend(loc="upper right")
    ax.set_title(title or spec.name)

    if savepath:
        fig.savefig(savepath, dpi=180, bbox_inches="tight")
    return fig, ax

# Plot a few family members
for name in ["P0-A-36R", "P1-C-48H", "P2-C2-144", "P3-CX-240"]:
    spec = next(s for s in family if s.name == name)
    geom = geoms_by_name[name]
    fig, ax = plot_geometry(spec, geom, title=f"{name} geometry", savepath=OUT_DIR / f"{name}_geometry.png")
    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# 9. Sweep engine across many sizes and variations
# ============================================================
def make_candidate_name(size_class: str, idx: int) -> str:
    return f"{size_class}-SWEEP-{idx:05d}"

def choose_battery_mass(diameter_m: float, architecture: str) -> float:
    if architecture == "cold_gas":
        return 0.04 + 0.30 * diameter_m
    if architecture == "compressor_fluidic":
        return 0.08 + 0.80 * diameter_m
    return 0.12 + 2.10 * diameter_m

def choose_avionics_mass(diameter_m: float) -> float:
    return 0.035 + 0.42 * diameter_m

def candidate_generator(max_candidates: int = 3500):
    idx = 0
    for d in SWEEP_CFG["diameters_m"]:
        for ldr in SWEEP_CFG["lobe_depth_ratio"]:
            for strr in SWEEP_CFG["section_thickness_ratio"]:
                for shr in SWEEP_CFG["section_height_ratio"]:
                    for nt in SWEEP_CFG["nozzle_types"]:
                        for arch in SWEEP_CFG["architectures"]:
                            for sec in SWEEP_CFG["sector_counts"]:
                                for split in SWEEP_CFG["lift_split_central"]:
                                    # Reject invalid architecture/split combinations
                                    if arch != "hybrid_central_lift" and split > 0.0:
                                        continue
                                    if arch == "hybrid_central_lift" and split <= 0.0:
                                        continue
                                    if arch == "cold_gas" and d > 0.40:
                                        continue  # keep cold-gas at smaller scales
                                    if arch == "compressor_fluidic" and d > 0.80:
                                        continue  # pure fluidic gets very hard at larger sizes

                                    for dn in SWEEP_CFG["round_nozzle_diams_mm"]:
                                        for sl in SWEEP_CFG["slot_lengths_mm"]:
                                            for sw in SWEEP_CFG["slot_widths_mm"]:
                                                for pk in SWEEP_CFG["plenum_pressure_kpa_gauge"]:
                                                    size_class = size_class_from_diameter(d)

                                                    # Belt balance heuristics by architecture
                                                    if arch == "hybrid_central_lift":
                                                        lb, ub, tb = 0.32, 0.18, 0.50
                                                    elif arch == "compressor_fluidic":
                                                        lb, ub, tb = 0.42, 0.23, 0.35
                                                    else:
                                                        lb, ub, tb = 0.38, 0.28, 0.34

                                                    spec = PrototypeSpec(
                                                        name=make_candidate_name(size_class, idx),
                                                        size_class=size_class,
                                                        architecture=arch,
                                                        diameter_m=d,
                                                        lobe_depth_ratio=ldr,
                                                        section_thickness_ratio=strr,
                                                        section_height_ratio=shr,
                                                        nozzle_type=nt,
                                                        sector_count=sec,
                                                        lower_belt_fraction=lb,
                                                        upper_belt_fraction=ub,
                                                        tangential_belt_fraction=tb,
                                                        round_nozzle_diam_mm=dn,
                                                        slot_length_mm=sl,
                                                        slot_width_mm=sw,
                                                        plenum_pressure_kpa_gauge=pk,
                                                        lift_split_central=split,
                                                        battery_mass_kg=choose_battery_mass(d, arch),
                                                        avionics_mass_kg=choose_avionics_mass(d),
                                                        structure_mass_factor=1.0,
                                                        payload_mass_kg=0.0 if size_class in ["P0", "P1"] else (0.15 if size_class=="P2" else 1.0),
                                                    )
                                                    yield spec
                                                    idx += 1
                                                    if idx >= max_candidates:
                                                        return

sweep_rows = []
t0 = time.time()
for i, spec in enumerate(candidate_generator(max_candidates=2800), start=1):
    geom, perf = evaluate_prototype(spec)
    row = {**asdict(spec), **asdict(perf)}
    for k, v in perf.notes.items():
        row[k] = v
    sweep_rows.append(row)

sweep_df = pd.DataFrame(sweep_rows)
elapsed = time.time() - t0
print("Candidates evaluated:", len(sweep_df), "in", round(elapsed, 2), "sec")
sweep_df.head()

In [ ]:
# ============================================================
# 10. Rank and filter sweep results
# ============================================================
# Useful filters: hover-capable and not absurdly low endurance.
feasible_df = sweep_df[(sweep_df["hover_margin"] >= 1.02) & (sweep_df["endurance_min"] >= 2.0)].copy()

# Size-wise winners
top_by_size = (
    feasible_df.sort_values("total_score", ascending=False)
    .groupby("size_class", as_index=False)
    .head(12)
    .reset_index(drop=True)
)

global_top = feasible_df.sort_values("total_score", ascending=False).head(40).reset_index(drop=True)

print("Total feasible:", len(feasible_df))
print("\nTop by size:")
display(top_by_size[[
    "name","size_class","architecture","diameter_m","nozzle_type","sector_count",
    "plenum_pressure_kpa_gauge","lift_split_central","gross_mass_kg","hover_margin",
    "endurance_min","control_authority_score","total_score"
]].round(3))

print("\nGlobal top:")
display(global_top[[
    "name","size_class","architecture","diameter_m","nozzle_type","sector_count",
    "plenum_pressure_kpa_gauge","lift_split_central","gross_mass_kg","hover_margin",
    "endurance_min","control_authority_score","total_score"
]].round(3))

In [ ]:
# Save sweep tables
sweep_csv = OUT_DIR / "sweep_results.csv"
feasible_csv = OUT_DIR / "sweep_feasible_results.csv"
top_csv = OUT_DIR / "sweep_top_by_size.csv"

sweep_df.to_csv(sweep_csv, index=False)
feasible_df.to_csv(feasible_csv, index=False)
top_by_size.to_csv(top_csv, index=False)

print("Saved sweep outputs.")
print(sweep_csv)
print(feasible_csv)
print(top_csv)

In [ ]:
# ============================================================
# 11. Visualization of trade space
# ============================================================
def scatter_trade(df: pd.DataFrame, x: str, y: str, by: str, title: str, savepath: Optional[pathlib.Path] = None):
    fig, ax = plt.subplots(figsize=(10, 7))
    groups = list(df[by].dropna().unique())
    for g in groups:
        sub = df[df[by] == g]
        ax.scatter(sub[x], sub[y], s=28, alpha=0.75, label=str(g))
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend()
    if savepath:
        fig.savefig(savepath, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)

scatter_trade(
    feasible_df,
    x="endurance_min",
    y="control_authority_score",
    by="architecture",
    title="Endurance vs control authority",
    savepath=OUT_DIR / "trade_endurance_vs_control.png",
)

scatter_trade(
    feasible_df,
    x="gross_mass_kg",
    y="hover_margin",
    by="size_class",
    title="Gross mass vs hover margin",
    savepath=OUT_DIR / "trade_mass_vs_hover_margin.png",
)

scatter_trade(
    feasible_df,
    x="sector_count",
    y="yaw_authority_score",
    by="nozzle_type",
    title="Sector count vs yaw authority",
    savepath=OUT_DIR / "trade_sector_vs_yaw.png",
)

In [ ]:
# ============================================================
# 12. Best-per-architecture summary
# ============================================================
best_arch = (
    feasible_df.sort_values("total_score", ascending=False)
    .groupby("architecture", as_index=False)
    .head(10)
    .reset_index(drop=True)
)

summary_cols = [
    "name","size_class","architecture","diameter_m","nozzle_type","sector_count",
    "plenum_pressure_kpa_gauge","lift_split_central","gross_mass_kg","thrust_total_N",
    "hover_margin","hover_power_W","endurance_min","control_authority_score",
    "yaw_authority_score","fault_tolerance_score","manufacturability_score","practical_score","total_score"
]
display(best_arch[summary_cols].round(3))

best_arch.to_csv(OUT_DIR / "best_by_architecture.csv", index=False)

In [ ]:
# ============================================================
# 13. Reconstruct top geometries for inspection
# ============================================================
def row_to_spec(row: pd.Series) -> PrototypeSpec:
    keys = PrototypeSpec.__dataclass_fields__.keys()
    return PrototypeSpec(**{k: row[k] for k in keys})

top_geom_dir = OUT_DIR / "top_geometries"
top_geom_dir.mkdir(exist_ok=True, parents=True)

for _, row in global_top.head(8).iterrows():
    spec = row_to_spec(row)
    geom = generate_tri_lobed_geometry(
        diameter_m=spec.diameter_m,
        lobe_depth_ratio=spec.lobe_depth_ratio,
        section_thickness_ratio=spec.section_thickness_ratio,
        section_height_ratio=spec.section_height_ratio,
        sector_count=spec.sector_count,
        lower_belt_fraction=spec.lower_belt_fraction,
        upper_belt_fraction=spec.upper_belt_fraction,
        tangential_belt_fraction=spec.tangential_belt_fraction,
    )
    fig, ax = plot_geometry(spec, geom, title=f"{spec.name} | score={row['total_score']:.1f}", savepath=top_geom_dir / f"{spec.name}.png")
    plt.show()
    plt.close(fig)

print("Saved geometry previews to:", top_geom_dir)

In [ ]:
# ============================================================
# 14. Recommended winners and engineering readout
# ============================================================
def choose_recommended(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for size in ["P0", "P1", "P2", "P3"]:
        sub = df[df["size_class"] == size].sort_values("total_score", ascending=False)
        if len(sub):
            out.append(sub.iloc[0])
    return pd.DataFrame(out)

recommended_df = choose_recommended(feasible_df)

display(recommended_df[[
    "name","size_class","architecture","diameter_m","nozzle_type","sector_count",
    "plenum_pressure_kpa_gauge","lift_split_central","gross_mass_kg","hover_margin",
    "endurance_min","control_authority_score","yaw_authority_score","total_score"
]].round(3))

recommended_df.to_csv(OUT_DIR / "recommended_winners.csv", index=False)

def print_engineering_readout(df: pd.DataFrame):
    for _, r in df.iterrows():
        print("=" * 88)
        print(f"{r['size_class']} winner: {r['name']}")
        print(f"  architecture        : {r['architecture']}")
        print(f"  diameter [m]        : {r['diameter_m']:.3f}")
        print(f"  nozzle type         : {r['nozzle_type']}")
        print(f"  sector count        : {int(r['sector_count'])}")
        print(f"  plenum pressure kPa : {r['plenum_pressure_kpa_gauge']:.1f}")
        print(f"  lift split central  : {r['lift_split_central']:.2f}")
        print(f"  gross mass [kg]     : {r['gross_mass_kg']:.3f}")
        print(f"  total thrust [N]    : {r['thrust_total_N']:.2f}")
        print(f"  hover margin        : {r['hover_margin']:.3f}")
        print(f"  endurance [min]     : {r['endurance_min']:.2f}")
        print(f"  control authority   : {r['control_authority_score']:.2f}")
        print(f"  yaw authority       : {r['yaw_authority_score']:.2f}")
        print(f"  practical score     : {r['practical_score']:.2f}")
        print(f"  total score         : {r['total_score']:.2f}")

print_engineering_readout(recommended_df)

In [ ]:
# ============================================================
# 15. Simple what-if editing cell
# ============================================================
# Duplicate this block and edit any field you want.
custom_spec = PrototypeSpec(
    name="CUSTOM-EDIT-ME",
    size_class=size_class_from_diameter(0.62),
    architecture="hybrid_central_lift",
    diameter_m=0.62,
    lobe_depth_ratio=0.22,
    section_thickness_ratio=0.14,
    section_height_ratio=0.12,
    nozzle_type="mixed",
    sector_count=48,
    lower_belt_fraction=0.40,
    upper_belt_fraction=0.18,
    tangential_belt_fraction=0.42,
    round_nozzle_diam_mm=0.8,
    slot_length_mm=5.0,
    slot_width_mm=0.5,
    plenum_pressure_kpa_gauge=25.0,
    lift_split_central=0.80,
    battery_mass_kg=1.55,
    avionics_mass_kg=0.25,
    payload_mass_kg=0.65,
)

custom_geom, custom_perf = evaluate_prototype(custom_spec)
print(json.dumps(asdict(custom_perf), indent=2))

fig, ax = plot_geometry(custom_spec, custom_geom, title="Custom prototype geometry", savepath=OUT_DIR / "custom_geometry.png")
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# 16. Export bundle ZIP
# ============================================================
manifest = {
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "out_dir": str(OUT_DIR.resolve()),
    "files": sorted([str(p.name) for p in OUT_DIR.rglob("*") if p.is_file()]),
    "recommended_winners": recommended_df.to_dict(orient="records"),
}

manifest_path = OUT_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

zip_path = pathlib.Path("tri_lobed_ring_drone_family_bundle.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(OUT_DIR.parent)))

print("Bundle written:", zip_path.resolve())
print("Manifest:", manifest_path.resolve())

## Notes on interpretation

This notebook gives you a **design-space exploration environment**, not validated CFD or flight certification.

Treat it as:

- a fast way to compare families,
- a way to identify likely winners,
- a way to export structured results,
- a scaffold for the next layer:
  - CFD,
  - bench rig force testing,
  - control-allocation simulation,
  - hardware BOM synthesis,
  - CAD refinement.

## Best practical reading of results

In most runs, the **hybrid central-lift + ring micro-nozzle** variants should dominate the top of the ranking because:

- pure fluidic lift suffers on hover efficiency,
- hybrid architecture preserves the agility and novelty of the ring,
- the tri-lobed geometry gives strong distributed moment arms and yaw leverage.

## Next expansions you can add

Good next cells to add after this notebook:
1. higher-fidelity disk loading and motor sizing,
2. Monte Carlo fault injection by disabled sectors,
3. nonlinear control allocation,
4. tethered rig dynamics,
5. body-frame 6-DOF simulation,
6. export to CAD-ready DXF/SVG section views.